In [41]:
# Importações necessárias do PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lit

# 1. Inicialização da Sessão Spark
spark = SparkSession.builder \
    .appName("Telecom-PDD-Silver-Processing") \
    .getOrCreate()

# Definindo caminhos 
raw_path = "s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/01-raw/"

silver_path = "s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/02-trusted/"


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [42]:
# 2. Leitura dos CSVs brutos e criação das views temporárias iniciais
df_clientes = spark.read.option("header", "true").csv("s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/01-raw/tb_cliente/")

df_faturas = spark.read.option("header", "true").csv("s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/01-raw/tb_fatura/")

df_pagamentos = spark.read.option("header", "true").csv("s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/01-raw/tb_pagamento/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [43]:
df_clientes.createOrReplaceTempView("df_clientes")

df_faturas.createOrReplaceTempView("df_faturas")

df_pagamentos.createOrReplaceTempView("df_pagamentos")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [44]:
qtd_registros = df_clientes.count()
print(f'Clientes - Quantidade de Registros: {qtd_registros}')

qtd_registros = df_faturas.count()
print(f'Faturas - Quantidade de Registros: {qtd_registros}')

qtd_registros = df_pagamentos.count()
print(f'Pagamentos - Quantidade de Registros: {qtd_registros}')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Clientes - Quantidade de Registros: 1500
Faturas - Quantidade de Registros: 10000
Pagamentos - Quantidade de Registros: 8496

In [45]:
df_clientes.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- cliente_id: string (nullable = true)
 |-- plano: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- score_credito: string (nullable = true)

In [46]:
# 3. Transformações e tipagens usando Spark SQL
df_clientes_silver = spark.sql("""
    SELECT 
        cliente_id,
        plano,
        regiao,
        score_credito
    FROM df_clientes
""")

df_clientes_silver.show(6)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-------------+-------+-------------+
|cliente_id|        plano| regiao|score_credito|
+----------+-------------+-------+-------------+
| CLI_00001|  Fibra 300MB|    Sul|          612|
| CLI_00002|Controle 15GB|Sudeste|          818|
| CLI_00003|Pós-Pago 50GB|Sudeste|          505|
| CLI_00004|  Fibra 500MB|Sudeste|          674|
| CLI_00005|  Fibra 300MB|Sudeste|          750|
| CLI_00006|  Fibra 300MB|Sudeste|          925|
+----------+-------------+-------+-------------+
only showing top 6 rows

In [47]:
df_faturas.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- fatura_id: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- data_emissao: string (nullable = true)
 |-- data_vencimento: string (nullable = true)
 |-- valor_fatura: string (nullable = true)
 |-- ano: string (nullable = true)
 |-- mes: string (nullable = true)

In [48]:
df_faturas_silver = spark.sql("""
    SELECT 
        fatura_id,
        cliente_id,
        to_date(data_emissao) as data_emissao,
        to_date(data_vencimento) as data_vencimento,
        cast(valor_fatura as decimal(14,2)) as valor_fatura,
        cast(ano as int) as ano,
        lpad(mes, 2, '0') as mes
    FROM df_faturas
""")

df_faturas_silver.show(4)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+----------+------------+---------------+------------+----+---+
| fatura_id|cliente_id|data_emissao|data_vencimento|valor_fatura| ano|mes|
+----------+----------+------------+---------------+------------+----+---+
|FAT_000001| CLI_01296|  2024-08-06|     2024-08-21|       99.90|2024| 08|
|FAT_000002| CLI_01078|  2026-01-28|     2026-02-12|      129.90|2026| 02|
|FAT_000003| CLI_00120|  2025-06-11|     2025-06-26|      129.90|2025| 06|
|FAT_000004| CLI_00147|  2024-04-14|     2024-04-29|      129.90|2024| 04|
+----------+----------+------------+---------------+------------+----+---+
only showing top 4 rows

In [49]:
df_pagamentos.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- fatura_id: string (nullable = true)
 |-- data_pagamento: string (nullable = true)

In [50]:
df_pagamentos.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+--------------+
| fatura_id|data_pagamento|
+----------+--------------+
|FAT_000001|    23/08/2024|
|FAT_000003|    26/06/2025|
|FAT_000004|    29/04/2024|
|FAT_000005|    05/09/2024|
|FAT_000006|    22/04/2025|
|FAT_000007|    14/06/2024|
|FAT_000008|    25/05/2024|
|FAT_000009|    29/11/2025|
|FAT_000010|    17/09/2025|
|FAT_000011|    25/11/2025|
|FAT_000013|    03/01/2025|
|FAT_000014|    04/08/2024|
|FAT_000015|    08/02/2026|
|FAT_000016|    11/02/2026|
|FAT_000017|    07/03/2025|
|FAT_000018|    15/01/2026|
|FAT_000019|    12/07/2025|
|FAT_000020|    16/12/2025|
|FAT_000022|    16/08/2025|
|FAT_000023|    24/11/2025|
+----------+--------------+
only showing top 20 rows

In [51]:
df_pagamentos_silver = spark.sql("""
    SELECT 
        fatura_id,
        to_date(data_pagamento,'dd/mm/yyyy') as data_pagamento
    FROM df_pagamentos
""")

df_pagamentos_silver.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+--------------+
| fatura_id|data_pagamento|
+----------+--------------+
|FAT_000001|    2024-01-23|
|FAT_000003|    2025-01-26|
|FAT_000004|    2024-01-29|
|FAT_000005|    2024-01-05|
|FAT_000006|    2025-01-22|
+----------+--------------+
only showing top 5 rows

In [52]:
# 4. Escrita na Camada Silver (Parquet)

df_clientes_silver.write.mode("overwrite").parquet(f"{silver_path}01-clientes/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [53]:
df_faturas_silver.write \
    .mode("overwrite") \
    .partitionBy("ano", "mes") \
    .parquet(f"{silver_path}02-faturas/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [54]:
df_pagamentos_silver.write.mode("overwrite").parquet(f"{silver_path}03-pagamentos/")

print("Processamento da camada Silver concluído com Spark SQL!")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Processamento da camada Silver conclu?do com Spark SQL!